In [1]:
import os, json, sqlite3
from datetime import date, datetime
from dotenv import load_dotenv
from openai import OpenAI
from PIL import Image, ImageDraw, ImageFont
import gradio

c:\Users\Kaweesh\Desktop\Project\sample-ai-agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv(override=True)
MODEL = "gpt-4.1-mini"
openai = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

### system prompt

In [3]:
HOSPITAL_NAME = "Lanka Care Hospital"
DB_NAME = "hospital.db"

system_message = f"""
You are a friendly, caring virtual assistant for {HOSPITAL_NAME}.
Reply in the same language the patient uses (English, Sinhala or Singlish). Keep answers short, warm and polite, 2 to 3 sentences.
Today's date is {date.today()} ({date.today().strftime('%A')}).

You can: register new patients, show doctor details, channel (book) a doctor, cancel a channeling, and explain a patient's test report.

STRICT RULES:
- NEVER recommend, suggest, name or give doses of any medicine or treatment to anyone. If asked, politely say you cannot advise on medicine and that they should consult a doctor, and offer to channel one.
- NEVER diagnose. For test reports only read out the stored results and notes; tell the patient to discuss them with their doctor.
- Never invent doctors, dates, prices or results. Use the tools; if unsure, say so.
- If the patient describes serious symptoms (chest pain, difficulty breathing, heavy bleeding, etc.), tell them to go to the emergency unit immediately or call 1990.

Flows:
- New patient: ask name, phone, age, gender. Then call register_patient. Tell them their Patient ID and where the card is saved.
- Channel: ask Patient ID (or offer to register), which doctor or specialty, and the date (YYYY-MM-DD). Use get_doctors to show options first. Then call book_appointment, and tell them the appointment ID, appointment number, fee and receipt location.
- Cancel: ask appointment ID and the phone number, then call cancel_appointment.
- Test report: ask Patient ID and registered phone number, then call get_test_reports.
"""


### DATABASE

In [4]:
def init_db():
    conn = sqlite3.connect(DB_NAME)
    conn.executescript("""
    CREATE TABLE IF NOT EXISTS patients (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT, phone TEXT, age INTEGER, gender TEXT, registered_on TEXT
    );
    CREATE TABLE IF NOT EXISTS doctors (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT, specialty TEXT, qualifications TEXT,
        available_days TEXT, time_slot TEXT, room TEXT, fee INTEGER
    );
    CREATE TABLE IF NOT EXISTS appointments (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        patient_id INTEGER, doctor_id INTEGER, appt_date TEXT,
        appt_number INTEGER, fee INTEGER, status TEXT
    );
    CREATE TABLE IF NOT EXISTS reports (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        patient_id INTEGER, test_name TEXT, test_date TEXT,
        result TEXT, notes TEXT
    );
    """)
    # dummy data (only once)
    if conn.execute("SELECT COUNT(*) FROM doctors").fetchone()[0] == 0:
        conn.executemany(
            "INSERT INTO doctors (name, specialty, qualifications, available_days, time_slot, room, fee) VALUES (?,?,?,?,?,?,?)",
            [
                ("Dr. Sunil Fernando", "Cardiologist", "MBBS, MD, MRCP", "Monday,Wednesday,Friday", "4:00 PM - 7:00 PM", "Room 101", 3500),
                ("Dr. Anjali Perera", "Pediatrician", "MBBS, DCH, MD (Paeds)", "Tuesday,Thursday,Saturday", "9:00 AM - 12:00 PM", "Room 205", 3000),
                ("Dr. Ruwan Silva", "General Physician", "MBBS, MD", "Monday,Tuesday,Wednesday,Thursday,Friday", "5:00 PM - 8:00 PM", "Room 110", 2000),
                ("Dr. Nadeesha Jayasinghe", "Dermatologist", "MBBS, MD (Derm)", "Wednesday,Saturday", "2:00 PM - 5:00 PM", "Room 302", 3200),
                ("Dr. Kamal Wijesinghe", "Orthopedic Surgeon", "MBBS, MS (Ortho), FRCS", "Monday,Thursday", "3:00 PM - 6:00 PM", "Room 208", 4000),
            ])
        conn.execute("INSERT INTO patients (name, phone, age, gender, registered_on) VALUES (?,?,?,?,?)",
                     ("Nimal Perera", "0771234567", 45, "Male", str(date.today())))
        conn.executemany(
            "INSERT INTO reports (patient_id, test_name, test_date, result, notes) VALUES (?,?,?,?,?)",
            [
                (1, "Full Blood Count (FBC)", "2026-10-20",
                 "Hemoglobin 13.8 g/dL, WBC 6,500 /uL, Platelets 250,000 /uL", "All values within normal range."),
                (1, "Fasting Blood Sugar", "2026-10-20",
                 "108 mg/dL", "Slightly above normal range (70-100 mg/dL). Please discuss with your doctor."),
                (1, "Lipid Profile", "2026-10-21",
                 "Total Cholesterol 215 mg/dL, LDL 140 mg/dL, HDL 45 mg/dL", "Borderline high. Please consult your doctor."),
            ])
    conn.commit()
    conn.close()

init_db()

def pid(n):  # patient code like PT0001
    return f"PT{int(n):04d}"

def parse_pid(p):
    return int(str(p).upper().replace("PT", "").strip())


### TOOLS

In [5]:
def register_patient(name, phone, age, gender):
    conn = sqlite3.connect(DB_NAME)
    existing = conn.execute("SELECT id FROM patients WHERE phone = ?", (phone,)).fetchone()
    if existing:
        conn.close()
        return f"This phone number is already registered. Patient ID is {pid(existing[0])}."
    cur = conn.execute(
        "INSERT INTO patients (name, phone, age, gender, registered_on) VALUES (?,?,?,?,?)",
        (name, phone, age, gender, str(date.today())))
    conn.commit()
    patient_id = cur.lastrowid
    conn.close()

    path = f"cards/patient_card_{pid(patient_id)}.png"
    draw_card("PATIENT CARD",
              [f"Patient ID : {pid(patient_id)}", f"Name       : {name}",
               f"Age/Gender : {age} / {gender}", f"Phone      : {phone}",
               f"Registered : {date.today()}"],
              path, footer="Please bring this card on every visit.")
    return f"Registered. Patient ID is {pid(patient_id)}. Patient card saved at {path}"

def get_doctors(specialty=None, name=None):
    conn = sqlite3.connect(DB_NAME)
    q, params = "SELECT id, name, specialty, qualifications, available_days, time_slot, room, fee FROM doctors WHERE 1=1", []
    if specialty:
        q += " AND LOWER(specialty) LIKE ?"; params.append(f"%{specialty.lower()}%")
    if name:
        q += " AND LOWER(name) LIKE ?"; params.append(f"%{name.lower()}%")
    rows = conn.execute(q, params).fetchall()
    conn.close()
    if not rows:
        return "No doctors found for that search."
    return "\n".join(
        f"[Doctor ID {r[0]}] {r[1]} - {r[2]} ({r[3]}). Days: {r[4]}. Time: {r[5]}. {r[6]}. Fee: Rs. {r[7]}"
        for r in rows)

def book_appointment(patient_id, doctor_id, appt_date):
    try:
        p_id = parse_pid(patient_id)
        day = datetime.strptime(appt_date, "%Y-%m-%d")
    except ValueError:
        return "Invalid patient ID or date format. Date must be YYYY-MM-DD."
    if day.date() < date.today():
        return "That date is in the past. Please choose a future date."

    conn = sqlite3.connect(DB_NAME)
    patient = conn.execute("SELECT name, phone FROM patients WHERE id = ?", (p_id,)).fetchone()
    doctor = conn.execute("SELECT name, specialty, available_days, time_slot, room, fee FROM doctors WHERE id = ?",
                          (doctor_id,)).fetchone()
    if not patient:
        conn.close(); return "Patient not found. The patient needs to register first."
    if not doctor:
        conn.close(); return "Doctor not found."
    weekday = day.strftime("%A")
    if weekday not in doctor[2].split(","):
        conn.close()
        return f"{doctor[0]} is not available on {weekday}. Available days: {doctor[2]}."

    count = conn.execute(
        "SELECT COUNT(*) FROM appointments WHERE doctor_id=? AND appt_date=? AND status='Confirmed'",
        (doctor_id, appt_date)).fetchone()[0]
    if count >= 20:
        conn.close(); return "Sorry, that doctor is fully booked on that date."
    appt_no = count + 1
    cur = conn.execute(
        "INSERT INTO appointments (patient_id, doctor_id, appt_date, appt_number, fee, status) VALUES (?,?,?,?,?,?)",
        (p_id, doctor_id, appt_date, appt_no, doctor[5], "Confirmed"))
    conn.commit()
    appt_id = cur.lastrowid
    conn.close()

    path = f"receipts/receipt_{appt_id}.png"
    draw_card("CHANNELING RECEIPT",
              [f"Appointment ID : {appt_id}", f"Patient : {patient[0]} ({pid(p_id)})",
               f"Doctor  : {doctor[0]}", f"Specialty : {doctor[1]}",
               f"Date    : {appt_date} ({weekday})", f"Time    : {doctor[3]}",
               f"Room    : {doctor[4]}", f"Appointment No : {appt_no}",
               f"FEE PAID : Rs. {doctor[5]}"],
              path, size=(600, 520))
    return (f"Appointment confirmed. Appointment ID {appt_id}, appointment number {appt_no}, "
            f"doctor {doctor[0]}, {appt_date} {doctor[3]}, {doctor[4]}, fee Rs. {doctor[5]}. Receipt saved at {path}")

def cancel_appointment(appointment_id, phone):
    conn = sqlite3.connect(DB_NAME)
    row = conn.execute(
        """SELECT a.id, a.status, a.appt_date, p.phone FROM appointments a
           JOIN patients p ON p.id = a.patient_id WHERE a.id = ?""", (appointment_id,)).fetchone()
    if not row:
        conn.close(); return "No appointment found with that ID."
    if row[3] != phone:
        conn.close(); return "The phone number does not match the one registered for this appointment."
    if row[1] == "Cancelled":
        conn.close(); return "This appointment is already cancelled."
    conn.execute("UPDATE appointments SET status='Cancelled' WHERE id=?", (appointment_id,))
    conn.commit(); conn.close()
    return f"Appointment {appointment_id} on {row[2]} has been cancelled."

def get_test_reports(patient_id, phone):
    try:
        p_id = parse_pid(patient_id)
    except ValueError:
        return "Invalid patient ID."
    conn = sqlite3.connect(DB_NAME)
    patient = conn.execute("SELECT name, phone FROM patients WHERE id=?", (p_id,)).fetchone()
    if not patient or patient[1] != phone:
        conn.close(); return "Verification failed. Patient ID and phone number do not match."
    rows = conn.execute("SELECT test_name, test_date, result, notes FROM reports WHERE patient_id=?",
                        (p_id,)).fetchall()
    conn.close()
    if not rows:
        return f"No test reports found for {patient[0]}."
    return f"Reports for {patient[0]}:\n" + "\n".join(
        f"- {r[0]} ({r[1]}): {r[2]}. Note: {r[3]}" for r in rows)

### TOOL SCHEMAS

In [6]:
def make_tool(name, desc, props, required):
    return {"type": "function", "function": {
        "name": name, "description": desc,
        "parameters": {"type": "object",
                       "properties": {k: {"type": t, "description": d} for k, (t, d) in props.items()},
                       "required": required, "additionalProperties": False}}}

tools = [
    make_tool("register_patient", "Register a new patient and generate a patient card.",
              {"name": ("string", "Full name"), "phone": ("string", "Phone number"),
               "age": ("integer", "Age"), "gender": ("string", "Male, Female or Other")},
              ["name", "phone", "age", "gender"]),
    make_tool("get_doctors", "Get doctor details. Filter by specialty or name, or leave empty for all doctors.",
              {"specialty": ("string", "e.g. Cardiologist, Pediatrician"), "name": ("string", "Doctor name")}, []),
    make_tool("book_appointment", "Channel a doctor for a patient and generate a receipt.",
              {"patient_id": ("string", "Patient ID like PT0001"), "doctor_id": ("integer", "Doctor ID"),
               "appt_date": ("string", "Date in YYYY-MM-DD")},
              ["patient_id", "doctor_id", "appt_date"]),
    make_tool("cancel_appointment", "Cancel an appointment after verifying the phone number.",
              {"appointment_id": ("integer", "Appointment ID"), "phone": ("string", "Registered phone number")},
              ["appointment_id", "phone"]),
    make_tool("get_test_reports", "Get a patient's test reports after verifying phone number.",
              {"patient_id": ("string", "Patient ID like PT0001"), "phone": ("string", "Registered phone number")},
              ["patient_id", "phone"]),
]

TOOL_FUNCS = {
    "register_patient": register_patient, "get_doctors": get_doctors,
    "book_appointment": book_appointment, "cancel_appointment": cancel_appointment,
    "get_test_reports": get_test_reports,
}

### Handel tools

In [7]:
def handle_tool_calls(message):
    responses = []
    for tc in message.tool_calls:
        print("Handling tool call:", tc.function.name)
        args = json.loads(tc.function.arguments)
        func = TOOL_FUNCS.get(tc.function.name)
        try:
            result = func(**args) if func else "Unknown tool"
        except Exception as e:
            result = f"Error: {e}"
        responses.append({"role": "tool", "content": result, "tool_call_id": tc.id})
    return responses

### Chat Function

In [8]:
def chat(message, history):
    cleaned = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + cleaned + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    while response.choices[0].finish_reason == "tool_calls":
        tool_message = response.choices[0].message
        messages.append(tool_message)
        messages.extend(handle_tool_calls(tool_message))
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    return response.choices[0].message.content

### Gradio Ui

In [9]:
gradio.ChatInterface(fn=chat, title=f"{HOSPITAL_NAME} Assistant").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Handling tool call: get_doctors
Handling tool call: get_doctors
Handling tool call: get_doctors
Handling tool call: register_patient
Handling tool call: register_patient
Handling tool call: get_doctors
Handling tool call: book_appointment
Handling tool call: book_appointment
Handling tool call: book_appointment
Handling tool call: book_appointment
Handling tool call: get_doctors
Handling tool call: book_appointment
